# Stacking Convolution - why depth matters

In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

### Lets first create a BLOCK i.e. 2 Conv (with ReLU) + a downsample.

In [4]:
# A Box

# 2 conv(with nonlinearity like ReLU)
conv1 = nn.Conv2d(in_channels = 3, out_channels = 64, kernel_size = 3, padding =1)
conv2 = nn.Conv2d(in_channels = 64, out_channels = 64, kernel_size =3, padding = 1)

# 1 Downsample

pool = nn.MaxPool2d(kernel_size = 2, stride = 2)

In [5]:
x = torch.rand(1,3, 224, 224)

print("input :", x.shape)

input : torch.Size([1, 3, 224, 224])


In [23]:
y = conv1(x)
print(y.shape)

torch.Size([1, 64, 224, 224])


In [31]:
# Using F = from torch's Neural Networks Library (functionality)
# ReLU = To have Non linearity i.e. Zeros out Negative values because No.s should not be negative ( 0 -> +ve)
y1 = F.relu(conv1(x))
print(y.shape)

torch.Size([1, 64, 224, 224])


In [32]:
y2 = conv2(y)
print(y2.shape)

torch.Size([1, 64, 224, 224])


In [33]:
y3 = F.relu(conv2(y1))
print(y3.shape)

torch.Size([1, 64, 224, 224])


In [34]:
y4 = pool(y3)
print(y4.shape)

torch.Size([1, 64, 112, 112])


After conv1: (1, 64, 224, 224) — channels go 3→64, spatial unchanged because padding=1 keeps it "same."

After conv2: (1, 64, 224, 224) — channels stay 64, spatial unchanged.

After pool: (1, 64, 112, 112) — spatial halved.



### Block 2: Stack four encoder stages

In [35]:
# Stage 1: 3 -> 64

s1_conv1 = nn.Conv2d(3, 64, kernel_size = 3, padding = 1 )
s1_conv2 = nn.Conv2d(64, 64, kernel_size = 3, padding = 1 )
s1_pool = nn.MaxPool2d(kernel_size = 2, stride = 2)

# Stage 2: 64 -> 128

s2_conv1 = nn.Conv2d(64, 128, kernel_size = 3, padding = 1 )
s2_conv2 = nn.Conv2d(128, 128, kernel_size = 3, padding = 1 )
s2_pool = nn.MaxPool2d(kernel_size = 2, stride = 2)

# Stage 3: 128 -> 256

s3_conv1 = nn.Conv2d(128, 256, kernel_size = 3, padding = 1 )
s3_conv2 = nn.Conv2d(256, 256, kernel_size = 3, padding = 1 )
s3_pool = nn.MaxPool2d(kernel_size = 2, stride =2)

# Stage 4: 256 -> 512
s4_conv1 = nn.Conv2d(256, 512, kernel_size = 3, padding = 1 )
s4_conv2 = nn.Conv2d(512, 512, kernel_size = 3, padding = 1 )



Stage 4 has no pool, by convention the bottom of the encoder doesn't downsample further.

In [43]:
z = torch.randn(1, 3, 224, 224)
print(z.shape)


torch.Size([1, 3, 224, 224])


In [44]:
z1 = F.relu(s1_conv1(z))
print(z1.shape)

torch.Size([1, 64, 224, 224])


In [45]:
z2 = F.relu(s1_conv2(z1))
z3 = s1_pool(z2)

print("Stage 1 :", z3.shape)

Stage 1 : torch.Size([1, 64, 112, 112])


In [46]:
y1 = F.relu(s2_conv1(z3))
y2 = F.relu(s2_conv2(y1))
y3 = s2_pool(y2)

print("Stage 2 :", y3.shape)

Stage 2 : torch.Size([1, 128, 56, 56])


In [47]:
x1 = F.relu(s3_conv1(y3))
x2 = F.relu(s3_conv2(x1))
x3 = s3_pool(x2)

print("Stage 3 :", x3.shape)

Stage 3 : torch.Size([1, 256, 28, 28])


In [48]:
w1 = F.relu(s4_conv1(x3))
w2 = F.relu(s4_conv2(w1))

print("Stage 4 :", w2.shape)

Stage 4 : torch.Size([1, 512, 28, 28])


Here we have increase the total representation size! While shrinking the spatial dimension.

We are trading "WHERE" for "WHAT"
The network knows less about exact spatial location but much more about what kind of patterns are present